  # DIY

  This notebooks shows how to custimize the heater/condenser and ohp configuration

  # Packages

  Firstly, let's import the necessary packages, you may need to install them
  for the first time.

In [ ]:
using ComputationalHeatTransfer # our main package
using Plots # for plotting
using ProgressMeter # to have a progress bar in the calculation

  # Specify properties

  ### Solid Physical parameters

  params is the HeatConductionParameters for the plate material. The numbers
  below represents aluminum.

In [ ]:
ρₛ = 2730; # material density [kg/m^3]
cₛ  = 8.93e02; # material specific heat [J/kg K]
kₛ  = 1.93e02; # material heat conductivity
plate_d = 6.35e-3; # effective d (The thickness of an ideal uniform thickness plate occupying the same volume)
params = HeatConductionParameters(ρₛ ,cₛ ,kₛ ,thickness=plate_d)

  ### Fluid Physical parameters

  pfluid contains the vapor and liquid properties at a constant reference
  temperature. Noted that the vapor pressure and the vapor density will be
  functions of temperatures during the simulation, other properties are
  extracted from pfluid as an approximate value.

In [ ]:
Tref = 291.2 # reference temperature
fluid_type = "Ammonia"
p_fluid = SaturationFluidProperty(fluid_type,Tref)

  # Set the geometries

  ### Geometry parameters

  The 2D domain is of rectangular shape (slightly different from ASETS-II). In
  the future it can be of arbitrary shape using the immersedlayers.jl package.

In [ ]:
Lx = 0.80 # Making it slightly larger to match your xlim plot bounds
Ly = 0.70 # Making it large enough to encompass the 0.60 max y

xlim = (-0.40, 0.40) # Changed to match block 
ylim = (-0.10, 0.60) # Changed to match block 

  ### Set mesh size and maximum time step for plate heat conduction

  Δx is controlled by Δx = α*gridPe and set having the same order of magitute
  of tube diameter 1e-3. Fourier number is used to give a safety "cap" of time
  step you can choose in the fluid module

In [ ]:
Δx, Δt_max = setstepsizes(params.α, gridPe=15.0, fourier=0.3)

  # Set up the evaporators and condensers

  In the "OHP simulation" notebook, I use "OHPtype" to look up a preset dictionary of OHP evaporators and condensers.

  You can also customize them, following the procedure below in this notebook.

  Firstly let's give the total heater power

In [ ]:
power = 50 # total heater power in watts

Then let's construct a heater

In [ ]:
# NEW DIMENSIONS APPLIED HERE
width_ohp = 0.3048
length_ohp = 0.5588

# Scaled up offset for the larger pipe geometry
x_offset = 0.03

Lheater_x = length_ohp * 0.10 
# Reduced from 1.15 to 1.05 to trim the overhang at the top
Lheater_y = width_ohp * 1.05

qe = power/Lheater_x/Lheater_y
eb1 = Rectangle(Lheater_x/2, Lheater_y/2, 1.5*Δx)

heater_center_x = (-length_ohp/2 + 0.01)*0
heater_center_y = width_ohp / 2 - 0.015 

Tfe = RigidTransform((heater_center_x, heater_center_y), 0.0)
Tfe(eb1)

eparams = [PrescribedHeatFluxRegion(qe,eb1)];

Then let's consctruct a condenser

In [ ]:
# Kept the length large to ensure full coverage
Lcondenser_x = length_ohp * 1.1
Lcondenser_y = width_ohp * 1.05

# Assuming a high-emissivity radiator coating (ε = 0.9) at Tref (291.2 K)
# 0.9 * 5.67e-8 * (291.2)^3
hc = 1.26 # condenser heat transfer coefficient

cb1 = Rectangle(Lcondenser_x/2, Lcondenser_y/2, 1.5*Δx)

# Shifted the center significantly to the left (from 0.075 to 0.02) to close the gap
condenser_center_x = 0 - 0.015

condenser_center_y = width_ohp / 2 - 0.0155 

Tfc = RigidTransform((condenser_center_x, condenser_center_y), 0.0)
Tfc(cb1)

Tc = Tref
cparams = [PrescribedHeatModelRegion(hc,Tc,cb1)];

# Set up OHP channel's shape

Similarly, In the "OHP simulation" notebook, I used **construct_ohp_curve("ASETS",Δx)** to look up a preset dictionary of ASETS-II OHP.

You can customize the ohp curve in either of the two ways:

 1. simply supply two arrays of x and y of the same length:

In [ ]:
a = 0.03
θ = 0:2π/1000:2π
r = a*sin.(2θ)
x = r .* cos.(θ)
y = r .* sin.(θ);

plot(x,y,aspectratio=1)

2. **construct_ohp_curve(nturn, pitch, height, gap, ds, x0, y0, flipx, flipy, angle)**, a built-in function to generate a closed loop multi-turn channel

In [ ]:
ds = 1.5Δx # point interval
nturn = 24 # number of turns

gap = 0.01 # gap between the closed loop end to the channel
pitch = width_ohp/(2*nturn+1) # pitch between channels
rotation_angle = π/2
x0, y0 = -length_ohp/2 * 1.02, -width_ohp/2 * 0.1 # starting point location

x,y = construct_ohp_curve(nturn,pitch,length_ohp,gap,ds,x0,y0,false,false,rotation_angle)
x = -x
y = y
plot(x,y,aspectratio=1)

In [ ]:
ohp = BasicBody(x,y) # build a BasicBody based on x,y
ohpgeom = ComputationalHeatTransfer.LineSourceParams(ohp) # build a line heat source based on BasicBody

  ### Plot what you got so far

  This is a exmaple of the compuational domain (the box) and the OHP channel
  serpentine (in blue)

In [ ]:
# Define limits to fit the new 0.635m x 0.4064m OHP dimensions
xlim = (-0.40, 0.40)
ylim = (-0.10, 0.60)

# plot ohp
plt = plot(ohp, fillalpha=0, linecolor=:black, xlims=xlim, ylims=ylim, framestyle=:box, xlabel="x [m]", ylabel="y [m]")

# plot heaters (red)
for ep in eparams
    plot!(ep)
end

# plot condensers (blue)
for cp in cparams
    plot!(cp)
end

# show plot
plt

  # Construct the systems

  ### Create HeatConduction system

  The solid module dealing with the 2D conduction, evaporator, condenser, and
  the OHP line heat source is constructed here.

In [ ]:
sys_plate = HeatConduction(params,Δx,xlim,ylim,Δt_max,qline=ohpgeom,qflux=eparams,qmodel=cparams)

  ### Create OHP inner channel system

  sys_tube: fluid module system

In [ ]:
sys_tube = initialize_ohpsys(sys_plate,p_fluid,power)

  # Initialize

  ### set time step

In [ ]:
tspan = (0.0, 30.0); # start time and end time
dt_record = 0.01   # saving time interval

tstep = 1e-3     # actrual time marching step

  ### combine inner tube and plate together

In [ ]:
u_plate = newstate(sys_plate) .+ Tref # initialize plate T field to uniform Tref
integrator_plate = init(u_plate,tspan,sys_plate) # construct integrator_plate

u_tube = newstate(sys_tube) # initialize OHP tube
integrator_tube = init(u_tube,tspan,sys_tube); # construct integrator_tube

  ### initialize arrays for saving

In [ ]:
SimuResult = SimulationResult(integrator_tube,integrator_plate);

  # Solve

  ### Run the simulation and store data

In [ ]:
@showprogress for t in tspan[1]:tstep:tspan[2]

    timemarching!(integrator_tube,integrator_plate,tstep)

    if (mod(integrator_plate.t,dt_record) < 1e-6) || (mod(-integrator_plate.t,dt_record) < 1e-6)
        store!(SimuResult,integrator_tube,integrator_plate)
    end

end

  # Store data

In [ ]:
save_path = "D:/CURRENT/solution_diy_24_turns_50W_30_sec.jld2"
save(save_path, "simulationResult", SimuResult)

In [ ]:
save_path = "D:/OHPDATA/solutiondiy34extendturnhundwatt.jld2"
save(save_path,"SimulationResult",SimuResult)

### take a peek at the solution (more at the PostProcessing notebook)

In [ ]:
@gif for i in eachindex(SimuResult.tube_hist_t)
    plot(OHPTemp(),i,SimuResult,clim=(291.2,294.0))
end

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*

In [ ]:
# 0. Define Time and Reference Parameters (brought over from your earlier cells)
Tref = 291.2         # reference temperature
tspan = (0.0, 7.0)   # start time and end time
dt_record = 0.01     # saving time interval
tstep = 1e-3         # actual time marching step

# 1. Define the list or range of turns you want to test
nturn_list = [25, 30, 40, 50, 60] 

# Begin the batch loop
for current_nturn in nturn_list
    println("--- Starting simulation for nturn = $current_nturn ---")

    # 2. Update geometry using current_nturn
    ds = 1.5 * Δx 
    gap = 0.01 
    
    # Calculate pitch dynamically based on the current turn count
    pitch = width_ohp / (2 * current_nturn + 1) 
    rotation_angle = π/2
    x0, y0 = -length_ohp/2 * 1.02, -width_ohp/2 * 0.1 

    x, y = construct_ohp_curve(current_nturn, pitch, length_ohp, gap, ds, x0, y0, false, false, rotation_angle)
    x = -x

    ohp = BasicBody(x, y) 
    ohpgeom = ComputationalHeatTransfer.LineSourceParams(ohp) 

    # 3. Construct the systems
    sys_plate = HeatConduction(params, Δx, xlim, ylim, Δt_max, qline=ohpgeom, qflux=eparams, qmodel=cparams)
    sys_tube = initialize_ohpsys(sys_plate, p_fluid, power)

    # 4. Initialize integrators
    u_plate = newstate(sys_plate) .+ Tref 
    integrator_plate = init(u_plate, tspan, sys_plate) 

    u_tube = newstate(sys_tube) 
    integrator_tube = init(u_tube, tspan, sys_tube)

    SimuResult = SimulationResult(integrator_tube, integrator_plate)

    # 5. Run the simulation
    @showprogress "Simulating $current_nturn turns..." for t in tspan[1]:tstep:tspan[2]
        timemarching!(integrator_tube, integrator_plate, tstep)
        
        if (mod(integrator_plate.t, dt_record) < 1e-6) || (mod(-integrator_plate.t, dt_record) < 1e-6)
            store!(SimuResult, integrator_tube, integrator_plate)
        end
    end

    # 6. Save with a DYNAMIC filename 
    save_path = "D:/OHPDATAEND/solution_diy_$(current_nturn)_turns_50W.jld2"
    save(save_path, "simulationResult", SimuResult)
    
    println("Finished and saved data to: $save_path\n")
end

println("All simulations completed!")